In [7]:
import os
from dotenv import load_dotenv, find_dotenv
from pymongo import MongoClient
import pandas as pd
from pymongo.server_api import ServerApi


# Load environment variables
load_dotenv(find_dotenv())

# Connect to MongoDB
mongo_uri = os.environ['MONGO_URI']
client = MongoClient(mongo_uri, server_api=ServerApi('1'))

# Select the database and collection
cm_env = os.environ.get('CM_ENV', 'dev')
if cm_env == 'prod':
    collection = client["campomaq"]["cm_ventas_prods"]
else:
    collection = client["campomaq_test"]["cm_ventas_prods"]

# Read all documents from the collection
cursor = collection.find({})
df_sales_prods = pd.DataFrame(list(cursor))


In [8]:
df_sales_prods = df_sales_prods[df_sales_prods['invoice_date'].dt.year > 2023]
df_sales_prods = df_sales_prods[~df_sales_prods["category_name"].str.lower().str.contains("servicios", na=False)]
df_sales_prods = df_sales_prods[~df_sales_prods["product_name"].str.lower().str.contains("varios", na=False)]


In [9]:
rules_df = pd.read_csv("product_rules.csv")

In [10]:
import pandas as pd
import numpy as np

# -------------------
# Step 1: Aggregate by product
# -------------------
df_grouped = df_sales_prods.groupby(
    ["product_code", "product_name", "brand_name", "category_name"], as_index=False
).agg({
    "quantity": "sum",
    "sale_without_iva": "sum",
    "total_cost": "sum"
})

# Profit
df_grouped["profit"] = df_grouped["sale_without_iva"] - df_grouped["total_cost"]

# Replace negative or zero profit with a small value
df_grouped["profit"] = df_grouped["profit"].apply(lambda x: x if x > 0 else 0.01)

# -------------------
# Step 2: Low-value and boost product flag
# -------------------
low_value_words = rules_df.query("rule_type == 'word' and category == 'low_value'")["rule_value"].str.lower().tolist()
low_value_codes = rules_df.query("rule_type == 'code' and category == 'low_value'")["rule_value"].tolist()

boost_words = rules_df.query("rule_type == 'word' and category == 'boost'")["rule_value"].str.lower().tolist()
boost_codes = rules_df.query("rule_type == 'code' and category == 'boost'")["rule_value"].tolist()

# -------------------
# Apply low-value flag
# -------------------
def is_low_value(row):
    name = row["product_name"].lower()
    code = row["product_code"]
    if any(word in name for word in low_value_words):
        return 1
    if code in low_value_codes:
        return 1
    return 0

# -------------------
# Apply strategic boost
# -------------------
def has_boost(row):
    name = row["product_name"].lower()
    code = row["product_code"]
    if any(word in name for word in boost_words):
        return 1
    if code in boost_codes:
        return 1
    return 0

df_grouped["main_boost"] = df_grouped.apply(has_boost, axis=1)
df_grouped["low_value_flag"] = df_grouped.apply(is_low_value, axis=1)


# -------------------
# Step 3: Normalization
# -------------------
# Quantity normalization
df_grouped["quantity_norm"] = (
    (df_grouped["quantity"] - df_grouped["quantity"].min()) /
    (df_grouped["quantity"].max() - df_grouped["quantity"].min())
)

# Penalize low-value items
df_grouped.loc[df_grouped["low_value_flag"] == 1, "quantity_norm"] *= 0.4

# Profit normalization
df_grouped["profit_norm"] = (
    (df_grouped["profit"] - df_grouped["profit"].min()) /
    (df_grouped["profit"].max() - df_grouped["profit"].min())
)


# -------------------
# Step 4: Popularity Index
# -------------------
df_grouped["popularity"] = (
    0.4 * df_grouped["quantity_norm"] +   # sales volume
    0.5 * df_grouped["profit_norm"] +     # profitability
    0.2 * df_grouped["main_boost"]        # strategic push
)

# -------------------
# Step 6: Sort products by popularity
# -------------------
df_grouped = df_grouped.sort_values("popularity", ascending=False)

# Check top results
# print(df_grouped[["product_name", "quantity", "profit", "popularity"]].head(15))


In [11]:
df_grouped.sort_values("popularity", ascending=False, inplace=True)
df_grouped.head(40)


,product_code,product_name,brand_name,category_name,quantity,sale_without_iva,total_cost,profit,main_boost,low_value_flag,quantity_norm,profit_norm,popularity
1127,408-0009SI,MOTOCULTOR TF545DE ARRANQUE ELECTRICO,HUSQVARNA,MOTOCULTORES,29,90930.000000,49013.093333,41916.906667,1,0,0.000306,1.000000,0.700122
1576,700-0001,MOTOCULTOR TKC-450,MITSUBISHI,MOTOCULTORES,8,43100.000000,21518.035814,21581.964186,1,0,0.000077,0.514875,0.457468
2394,700M-578,MOTOCULTOR TOPO Y LEVANTACAMAS,VARIOS MOTORES,MOTOCULTORES,3,13650.000000,10282.800000,3367.200000,1,0,0.000022,0.080330,0.240174
2326,700M-490SI,CUCHILLA CURVA IZQUIERDA APORCAR MOTOCULTOR TF...,HUSQVARNA,ACCESORIOS MOTOCULTORES,459,3701.345000,1044.225000,2657.120000,1,1,0.002002,0.063390,0.232496
961,311-0012SI,CILINDRO BOMBA 653/655,MARUYAMA,REPUESTOS FUMIGACION,123,13197.000000,10627.200000,2569.800000,1,0,0.001333,0.061307,0.231187
2328,700M-491SI,CUCHILLA CURVA DERECHA APORCAR MOTOCULTOR TF545DE,HUSQVARNA,ACCESORIOS MOTOCULTORES,445,3519.675000,1012.375000,2507.300000,1,1,0.001941,0.059816,0.230684
2610,S000088,ARTURITO CON BOMBA DE ACERO INOXIDABLE,VARIOS BOMBAS,MAQUINARIA FUMIGACION,1,2079.000000,261.300000,1817.700000,1,0,0.000000,0.043364,0.221682
2581,900-1001SI,MOTOCULTOR MULTIFUNCION 210CC DUCATI,AGROTA,MOTOCULTORES,4,7390.000000,5597.200000,1792.800000,1,0,0.000033,0.042770,0.221398
1158,408-0050,TRANSMISION MOTOCULTOR HUSQVARNA TF545DE,HUSQVARNA,REPUESTOS MOTOCULTORES,9,3284.088136,1628.442504,1655.645632,1,0,0.000087,0.039498,0.219784
997,313-0000SI,BOMBA FUMIGAR WHALE SB528,MARUYAMA,MAQUINARIA FUMIGACION,9,5248.410000,3766.500000,1481.910000,1,0,0.000087,0.035353,0.217712


In [ ]:
# if cm_env == 'prod':
#     collection = client["campomaq"]["cm_top_sales_products"]
# else:
#     collection = client["campomaq_test"]["cm_top_sales_products"]
    
# for record in df_grouped.to_dict(orient="records"):
#     # Use product_id as unique key, update if exists, insert if not
#     collection.update_one(
#         {"product_code": record["product_code"]},
#         {"$set": record},
#         upsert=True
#     )

# print(f"✅ Upserted {len(df_grouped)} products into MongoDB.")

In [12]:
if cm_env == 'prod':
    collection = client["campomaq"]["cm_top_sales_products"]
else:
    collection = client["campomaq_test"]["cm_top_sales_products"]

if not df_grouped.empty:
    # Convert DataFrame to JSON
    records = df_grouped.to_dict(orient="records")
    collection.insert_many(records)
    print(f"✅ Inserted {len(records)} records into MongoDB")

else:
    print("No new data to insert.")

✅ Inserted 2611 records into MongoDB
